# JobFit RAG Assistant — Agent Loop Test
Run this notebook in Colab (GPU runtime recommended).

Cells are organised by source file:
- Cell 1: Setup
- Cell 2: `llm.py` — SmolLM2 wrapper
- Cell 3: `similarity.py` — sentence embedding matching
- Cell 4: Tool functions
- Cell 5: Agent loop — `run_tool_workflow()`
- Cell 6: Test all 3 tools
- Cell 7: Edge cases

In [16]:
# Cell 1 — Setup
# !pip install transformers accelerate sentence-transformers scikit-learn -q

import torch
import json
import re
import os

print("GPU available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GPU available: True


In [17]:
# Cell 2 — llm.py (SmolLM2 wrapper)
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

_tokenizer = None
_model = None

def _load_model():
    global _tokenizer, _model
    if _model is None:
        print(f"Loading {MODEL_NAME}...")
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        _model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
        _model.eval()
        print("Model ready.")
    return _tokenizer, _model


def generate_text(prompt, temperature=0.7, top_k=50, top_p=0.9, max_new_tokens=200):
    tokenizer, model = _load_model()

    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("llm.py loaded")

llm.py loaded


In [18]:
# Cell 3 — similarity.py (sentence embeddings)
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

SIMILARITY_THRESHOLD = 0.50

def compute_similarity(resume_sentences, jd_sentences):
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    resume_embeddings = embedding_model.encode(resume_sentences)
    jd_embeddings = embedding_model.encode(jd_sentences)
    similarity_matrix = cosine_similarity(jd_embeddings, resume_embeddings)
    return similarity_matrix


def get_best_matches(resume_sentences, jd_sentences):
    matches = []
    similarity_matrix = compute_similarity(resume_sentences, jd_sentences)
    for i, jd in enumerate(jd_sentences):
        best_index = similarity_matrix[i].argmax()
        best_score = similarity_matrix[i][best_index]
        best_resume = resume_sentences[best_index]
        matches.append({
            "jd": jd,
            "resume": best_resume,
            "score": float(best_score)
        })
    return matches

print("similarity.py loaded")

similarity.py loaded


In [19]:
# Cell 4 — Tool functions (from tools.py + load_sentences from analyzer.py)

def load_sentences(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def calculate_match_score(resume_path, jd_path):
    resume_sentences = load_sentences(resume_path)
    jd_sentences = load_sentences(jd_path)
    matches = get_best_matches(resume_sentences, jd_sentences)
    avg_score = sum(m["score"] for m in matches) / len(matches) if matches else 0
    top = [m for m in matches if m["score"] >= SIMILARITY_THRESHOLD]
    return {"score": round(avg_score, 3), "top_matches": top}


def extract_skills(text):
    prompt = (
        "Extract all technical and professional skills from the text below. "
        "Return them as a comma-separated list. No explanation.\n\n"
        f"Text:\n{text}\n\nSkills:"
    )
    result = generate_text(prompt, temperature=0.2, max_new_tokens=100)
    skills = [s.strip().strip("[]\"'") for s in result.replace("\n", ",").split(",") if s.strip()]
    return skills if skills else []


def summarize_jd_llm(jd_text):
    prompt = (
        "Summarize this job description in 2-3 sentences. "
        "Focus on role, key responsibilities, and required skills.\n\n"
        f"Job Description:\n{jd_text}\n\nSummary:"
    )
    return generate_text(prompt, temperature=0.3, max_new_tokens=150)

print("Tool functions loaded")
print("  calculate_match_score(resume_path, jd_path)")
print("  extract_skills(text)")
print("  summarize_jd_llm(jd_text)")

Tool functions loaded
  calculate_match_score(resume_path, jd_path)
  extract_skills(text)
  summarize_jd_llm(jd_text)


In [20]:
# Cell 5 — Agent loop (function_caller.py)

TOOL_SCHEMAS = [
    {
        "name": "calculate_match_score",
        "description": "Calculates how well a resume matches a job description. Use when user asks about fit, match, or compatibility.",
        "parameters": {
            "type": "object",
            "properties": {
                "resume_path": {"type": "string", "description": "Path to resume file"},
                "jd_path": {"type": "string", "description": "Path to job description file"}
            },
            "required": ["resume_path", "jd_path"]
        }
    },
    {
        "name": "summarize_jd",
        "description": "Summarizes a job description into key points. Use when user wants a quick understanding of a JD.",
        "parameters": {
            "type": "object",
            "properties": {
                "jd_text": {"type": "string", "description": "Raw job description text"}
            },
            "required": ["jd_text"]
        }
    },
    {
        "name": "extract_skills",
        "description": "Extracts skills from any text (resume or JD). Use when user asks what skills are present or required.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Text to extract skills from"}
            },
            "required": ["text"]
        }
    }
]


TOOL_REGISTRY = {
    "calculate_match_score": calculate_match_score,
    "summarize_jd": summarize_jd_llm,
    "extract_skills": extract_skills,
}


_ALL_TOOL_KEYWORDS = {"match", "compare", "fit", "compatible",
                      "summarize", "summary", "overview", "summarise",
                      "skill", "extract"}


def parse_tool_call(response):
    cleaned = re.sub(r"```json|```", "", response).strip()
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group())
        if "tool" in parsed:
            return {"name": parsed["tool"], "arguments": parsed.get("parameters", parsed.get("arguments", {}))}
        if "name" in parsed:
            args = parsed.get("arguments", parsed.get("parameters", {}))
            return {"name": parsed["name"], "arguments": args}
        return None
    except json.JSONDecodeError:
        return None


def _keyword_fallback(query):
    q = query.lower()
    if any(w in q for w in ["match", "compare", "fit", "compatible"]):
        return "calculate_match_score"
    if any(w in q for w in ["summarize", "summary", "overview", "summarise"]):
        return "summarize_jd"
    if any(w in q for w in ["skill", "extract"]):
        return "extract_skills"
    return None


def run_tool_workflow(user_query, context=None, verbose=True):
    """
    context: dict with available data
        - resume_path: str  (for calculate_match_score)
        - jd_path: str      (for calculate_match_score)
        - jd_text: str      (for summarize_jd)
        - resume_text: str  (for extract_skills)
    """
    context = context or {}

    if verbose:
        print(f"\n{'='*50}")
        print(f"USER: {user_query}")
        print(f"{'='*50}")

    tool_context = json.dumps(TOOL_SCHEMAS, indent=2)
    decision_prompt = f"""You have access to these tools:
{tool_context}

RULES:
- If a tool matches the user's intent, respond ONLY with JSON.
- Use exactly this format:
{{"tool": "tool_name"}}
- Only pick the tool name, NOT the parameter values.
- If no tool is needed, answer the user directly in plain text.
- No explanation. No markdown.

User: {user_query}
"""

    llm_output = generate_text(decision_prompt, temperature=0.1, max_new_tokens=200)
    if verbose:
        print(f"\n[LLM] {llm_output}")

    tool_call = parse_tool_call(llm_output)
    routing = "llm"

    if not tool_call:
        tool_name = _keyword_fallback(user_query)
        if tool_name:
            routing = "keyword"
            if verbose:
                print(f"\n[Fallback] {tool_name}")
        else:
            direct_answer = generate_text(user_query, temperature=0.3, max_new_tokens=200)
        return {"answer": direct_answer, "tool": None, "routing": "direct"}
    else:
        tool_name = tool_call["name"]

    if verbose:
        print(f"\n[Tool] {tool_name} | Route: {routing}")

    if tool_name not in TOOL_REGISTRY:
        return {"answer": f"Unknown tool: {tool_name}", "tool": tool_name, "routing": routing}

    # Validate: LLM-picked tool should match query keywords
    if routing == "llm" and not any(kw in user_query.lower() for kw in _ALL_TOOL_KEYWORDS):
        if verbose:
            print(f"[Validate] No tool keywords in query -- answering directly")
        direct_answer = generate_text(user_query, temperature=0.3, max_new_tokens=200)
        return {"answer": direct_answer, "tool": None, "routing": "direct"}

    # Build real parameters from caller-provided context
    if tool_name == "calculate_match_score":
        params = {"resume_path": context.get("resume_path"), "jd_path": context.get("jd_path")}
    elif tool_name == "summarize_jd":
        params = {"jd_text": context.get("jd_text", "")}
    elif tool_name == "extract_skills":
        text = context.get("resume_text") or context.get("jd_text") or ""
        params = {"text": text}
    else:
        params = {}

    try:
        result = TOOL_REGISTRY[tool_name](**params)
        if verbose:
            print(f"[Result] {result}")
    except Exception as e:
        return {"answer": f"Error: {e}", "tool": tool_name, "routing": routing}

    # Skip final LLM call if result is already readable text
    if isinstance(result, str):
        final_answer = result
    else:
        final_prompt = f"""The user asked: "{user_query}"
You used tool: {tool_name}
Parameters: {params}
Result: {result}

Give a clear, short final answer based on the result."""
        final_answer = generate_text(final_prompt, temperature=0.3, max_new_tokens=150)

    if verbose:
        print(f"\n[Final] {final_answer}")

    return {"answer": final_answer, "tool": tool_name, "routing": routing}

print("function_caller.py loaded")

function_caller.py loaded


In [21]:
# Cell 6 — Test all 3 tools
# Create temp files for calculate_match_score to read

SAMPLE_RESUME = "Experienced software engineer skilled in Java, Python, C++, and data structures. Strong background in building scalable systems and leading engineering teams."
SAMPLE_JD = "We are looking for an AI Engineer with experience in Python, Machine Learning, Docker, FastAPI and MLOps. The candidate should have experience deploying models and building scalable ML systems."

with open("/tmp/resume.txt", "w") as f:
    f.write(SAMPLE_RESUME)
with open("/tmp/jd.txt", "w") as f:
    f.write(SAMPLE_JD)

sample_context = {
    "resume_path": "/tmp/resume.txt",
    "jd_path": "/tmp/jd.txt",
    "jd_text": SAMPLE_JD,
    "resume_text": SAMPLE_RESUME,
}

tests = [
    ("How well does my resume match this job?", sample_context),
    ("Summarize this job description for me", {"jd_text": SAMPLE_JD}),
    ("What skills are in this resume?", {"resume_text": SAMPLE_RESUME}),
]

for q, ctx in tests:
    out = run_tool_workflow(q, context=ctx, verbose=True)
    print(f"\nAnswer: {out['answer']}\n")
    print("-" * 50)


USER: How well does my resume match this job?
Loading HuggingFaceTB/SmolLM2-1.7B-Instruct...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Model ready.

[LLM] {"tool": "calculate_match_score"}

[Tool] calculate_match_score | Route: llm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Result] {'score': 0.566, 'top_matches': [{'jd': 'We are looking for an AI Engineer with experience in Python, Machine Learning, Docker, FastAPI and MLOps. The candidate should have experience deploying models and building scalable ML systems.', 'resume': 'Experienced software engineer skilled in Java, Python, C++, and data structures. Strong background in building scalable systems and leading engineering teams.', 'score': 0.5656954646110535}]}

[Final] Based on the provided resume and job description, your resume matches the job about 56.6% of the time.

Answer: Based on the provided resume and job description, your resume matches the job about 56.6% of the time.

--------------------------------------------------

USER: Summarize this job description for me

[LLM] {"tool": "summarize_jd"}

[Tool] summarize_jd | Route: llm
[Result] AI Engineer with experience in Python, Machine Learning, Docker, FastAPI, and MLOps is required to develop and deploy scalable ML systems. The role involve

In [22]:
# Cell 7 — Test edge cases (no-tool queries)
# The validation check should catch these and answer directly

edge_cases = [
    ("What is the capital of France?", {}),
    ("Hello, how are you?", {}),
]

for q, ctx in edge_cases:
    out = run_tool_workflow(q, context=ctx, verbose=True)
    print(f"\nAnswer: {out['answer']}\n")
    print("-" * 50)


USER: What is the capital of France?

[LLM] {"tool": "calculate_match_score"}

[Tool] calculate_match_score | Route: llm
[Validate] No tool keywords in query -- answering directly

Answer: The capital of France is Paris.

--------------------------------------------------

USER: Hello, how are you?

[LLM] {"tool": "calculate_match_score"}

[Tool] calculate_match_score | Route: llm
[Validate] No tool keywords in query -- answering directly

Answer: I'm doing well, thank you for asking! It's a pleasure to chat with you. I'm here to assist you with any questions or information you need. How can I help you today?

--------------------------------------------------
